## Imports

In [45]:
import os
import uuid
import asyncio
import dotenv
import openai
import pandas as pd
from datetime import datetime, timezone
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import instructor
from pydantic import BaseModel, Field

dotenv.load_dotenv()

True

## RAG Pipeline

In [46]:
class RAGUsedContext(BaseModel):
    id: int = Field("ID of the intervention")
    machine: str = Field("Machine of the intervention")
    date_start: str = Field("Date of the intervention")
    summary: str = Field("Summary of the intervention")

class RAGGenerationResponse(BaseModel):
    answer: str = Field("Answer to the question")
    references: list[RAGUsedContext] = Field("List of problems that answer the question")

In [47]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [48]:
qdrant_client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "cm_interventions"


def embed_text(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


def retrieve(query, top_k=5):
    vector = embed_text(query)
    hits = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=vector,
        limit=top_k,
    ).points
    return hits


def format_context(hits):
    parts = []
    for h in hits:
        p = h.payload
        parts.append(
            f"ID: {p.get('id')}\nMachine: {p.get('machine')}\n"
            f"Date: {p.get('date_start')}\nSummary: {p.get('summary')}"
        )
    return "\n\n---\n\n".join(parts)


RAG_PROMPT = """\
You are a maintenance assistant. Use the intervention records below to answer the question.
Be concise. Do not use markdown formatting.

Question: {question}

Records:
{context}
"""


def generate_answer(question, context):
    prompt = RAG_PROMPT.format(question=question, context=context)
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort":"low"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question, top_k=5):
    hits = retrieve(question, top_k)
    context = format_context(hits)
    answer = generate_answer(question, context)

    retrieved_ids = [h.payload.get('id') for h in hits]
    scores = [h.score for h in hits]

    final_result = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_ids,
        "retrieved_context": context,
        "similarity_score": scores
    }

    return final_result
    
    

In [49]:
question = "carry roller problems"
outputs = rag_pipeline(question)

/Users/jooaobrum/Library/CloudStorage/GoogleDrive-joao.paulo.brum14@gmail.com/My Drive/Projetos Pessoais/Projetos de Estudo/end2end-ai-engineering-bootcamp/hephaestus-agentic-maintenance/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:280: UserWarning: Qdrant client version 1.17.1 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


In [50]:
outputs

{'data_object': RAGGenerationResponse(answer='Carry roller problems summary from intervention records:\n\n1) CB-200 (INT-2024-0278) – Fault B-006 (2024-04-17)\n- Issue: Carry roller section 62 bearing alarm; elevated vibration and audible grinding.\n- Cause: Roller #62 bearing wear with early-stage spalling.\n- Action: Isolated/corrected root cause; replaced associated consumable; roller replaced; machine re-certified.\n- Result: Vibration nominal post-repair; monitoring interval increased for 4 weeks.\n\n2) CR-150/CR-100 roll subsystem notes (R-003 / R-006) – not specifically “carry roller”\n- CR-150 (INT-2024-0481) Fault R-003 (2024-06-22): WS work roll bearing high temperature due to spalling; bearing replaced; temp stabilised.\n- CR-100 (INT-2024-1014 and INT-2023-0530) Fault R-006: roll force deviation from seal wear (piston bypass); component replaced/cleaned & recalibrated; force balance returned.\n- CR-150 (INT-2024-0236) Fault R-006 (2024-04-05): force deviation from drive bel